# Đề tài 5: phân tích cảm xúc đánh giá sản phẩm thương mại điện tử (Tiki/Shopee)

In [1]:
pip install pandas numpy underthesea openpyxl

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached charset_normalizer-3.5.2-cp314-cp314-win_amd64.whl.metadata (47 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)
   ---------------------------------------- 0.0/7.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/7.3 MB ? eta -:--:--
   ---- ----------------------------------- 0.8/7.3 MB 2.1 MB/s eta 0:00:04
   ----- ---------------------------------- 1.0/7.3 MB 2.1 MB/s eta 0:00:03
   -------- ------------------------------- 1.6/7.3 MB 2.2 MB/s eta 0:00:03
   -------------- ------------------------- 2.6/7.3 MB 2.8 MB/s eta 0:00:02
   --------------- ------------------------ 2.9/7.3 MB 2.8 MB/s eta 0:00:02
   ----------------- ---------------------- 3.1/7.3 MB 2.6 MB/s eta 0:00:02
   ------------------ --------------------- 3.4/7.3 MB 2.4 MB/s eta 0:00:02
   -------------------- ------------------- 3.7/7.3 MB 2.2 MB/s eta 0:00:02
   --------------------- ----------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Lựa chọn dataset

### 1.1. Nguồn dữ liệu

Dataset được sử dụng là **Tiki Books Dataset**, trong đó file `comments.csv` chứa các đánh giá (bình luận + số sao) của khách hàng về sách trên sàn thương mại điện tử **Tiki**. Hai file còn lại (`book_data.csv`, `book_id.csv`) là thông tin sách, không dùng cho bài toán phân loại cảm xúc.

- Nguồn: Kaggle (dữ liệu được crawl từ Tiki) – đường dẫn dataset: **[ĐIỀN link trang dataset trên Kaggle]**
- Notebook tham khảo (chuẩn bị dữ liệu & EDA bộ Tiki Books): [biminhc/data-preparation-eda-tiki-books-dataset](https://www.kaggle.com/code/biminhc/data-preparation-eda-tiki-books-dataset)
- Giai đoạn thu thập: **[ĐIỀN, nếu có thông tin]**
- Ngôn ngữ: tiếng Việt

### 1.2. Số lượng mẫu

Dataset có **[ĐIỀN số dòng]** mẫu (số liệu thực tế được in ra ở mục 2.1).

### 1.3. Các thuộc tính

| Thuộc tính (tên gốc trong file) | Kiểu | Mô tả |
|---|---|---|
| **[ĐIỀN tên cột bình luận]** → đổi thành `text` | Chuỗi văn bản | Nội dung bình luận/đánh giá của khách hàng. |
| **[ĐIỀN tên cột số sao]** → đổi thành `rating` | Số nguyên 1–5 | Số sao khách hàng chấm cho sản phẩm. |
| **[ĐIỀN các cột còn lại, nếu có]** | | Ví dụ: mã sản phẩm, tên người dùng, thời gian… (sẽ bị loại ở mục 2.4 nếu không cần). |

### 1.4. Vì sao dataset phù hợp với đề tài

- **Đúng bối cảnh đề tài 5:** là đánh giá sản phẩm thương mại điện tử, văn bản tiếng Việt.
- **Có điểm đánh giá sao (1–5):** vừa dùng để gán nhãn cảm xúc (Positive/Negative/Neutral), vừa dùng để vẽ biểu đồ phân bố điểm sao (nhiệm vụ của Thành viên 2).
- **Quy mô đủ huấn luyện** ít nhất 2 mô hình phân loại (nhiệm vụ của các thành viên sau).

### 1.5. Cách gán nhãn cảm xúc từ số sao

| Số sao (`rating`) | Nhãn (`label`) | Cảm xúc |
|---|---|---|
| 1 – 2 sao | 0 | Negative (Tiêu cực) |
| 3 sao | 1 | Neutral (Trung lập) |
| 4 – 5 sao | 2 | Positive (Tích cực) |

### 1.6. Hạn chế của dataset

- **Mất cân bằng nhãn:** review thương mại điện tử thường chứa rất nhiều 5 sao, ít 3 sao, nên lớp Neutral thường rất nhỏ. **[ĐIỀN lại theo kết quả thật ở mục 3.2]**
- **Nhãn có nhiễu:** nhãn được suy ra từ số sao nên có thể lệch với nội dung, ví dụ khách chấm 5 sao nhưng bình luận phàn nàn, hoặc ngược lại.
- **Văn phong tự do:** nhiều bình luận ngắn, viết tắt/teencode ("ko", "dc", "sp"), có emoji, kéo dài chữ ("ngonnnn"), có thể có bình luận spam lặp lại.

## 2. Tiền xử lý dữ liệu (NumPy, Pandas)

### 2.0. Đọc dữ liệu

Cell dưới đây đọc file dữ liệu, tự dò hai cột quan trọng (bình luận và số sao) rồi đổi tên thành `text` và `rating`. Nếu tên cột trong file của nhóm khác các tên phổ biến, hãy gán thủ công `TEXT_COL` và `RATING_COL`.

In [3]:
import os
import re
import time
import unicodedata
import urllib.request
import numpy as np
import pandas as pd

PROJECT_ROOT = os.getcwd()
if not os.path.isdir(os.path.join(PROJECT_ROOT, "data")):
    PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)
CSV_PATH = os.path.join(PROJECT_ROOT, "data", "raw", "comments.csv")
TEXT_COL   = None
RATING_COL = None

TEXT_CANDIDATES   = ["text", "comment", "content", "review", "review_text", "reviews",
                     "noi_dung", "binh_luan", "body", "sentence", "nhan_xet", "nội dung", "bình luận"]
RATING_CANDIDATES = ["rating", "star", "stars", "score", "so_sao", "diem", "rate", "số sao", "điểm"]

def find_col(df, candidates, given=None, what=""):
    if given is not None:
        if given not in df.columns:
            raise KeyError(f"Không thấy cột '{given}'. Các cột hiện có: {list(df.columns)}")
        return given
    lower = {str(c).strip().lower(): c for c in df.columns}
    for cand in candidates:
        if cand in lower:
            return lower[cand]
    raise KeyError(f"Không tự dò được cột {what}. Các cột hiện có: {list(df.columns)}. "
                   f"Hãy gán thủ công TEXT_COL / RATING_COL.")

def read_any(path):
    if path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    for enc in ("utf-8-sig", "utf-8", "utf-16"):
        try:
            return pd.read_csv(path, encoding=enc)
        except (UnicodeDecodeError, UnicodeError):
            continue
    return pd.read_csv(path, encoding="latin-1")

if os.path.isdir(CSV_PATH):
    nested_csv_path = os.path.join(CSV_PATH, os.path.basename(CSV_PATH))
    if os.path.isfile(nested_csv_path):
        CSV_PATH = nested_csv_path

if not os.path.isfile(CSV_PATH):
    raise FileNotFoundError(f"Không tìm thấy file dữ liệu '{CSV_PATH}'. Hãy kiểm tra đường dẫn và tên file.")

# Thêm block bắt lỗi file bị khóa
try:
    df = read_any(CSV_PATH)
    print("Đọc file thành công!")
except PermissionError:
    print(f"\n[LỖI QUYỀN TRUY CẬP] Python không thể đọc file '{CSV_PATH}'.")
    print("-> Hãy kiểm tra quyền truy cập và xác nhận đây là file (không phải thư mục). Nếu file đang bị ứng dụng khác khóa, hãy đóng ứng dụng đó rồi chạy lại.\n")
    raise

print("Các cột trong file gốc:", list(df.columns))
text_col   = find_col(df, TEXT_CANDIDATES,   TEXT_COL,   "bình luận")
rating_col = find_col(df, RATING_CANDIDATES, RATING_COL, "số sao")
df = df.rename(columns={text_col: "text", rating_col: "rating"})

log = {"Số dòng ban đầu": len(df)}

print("Đường dẫn file:", os.path.abspath(CSV_PATH))
print(f"Cột bình luận: '{text_col}' -> text | Cột số sao: '{rating_col}' -> rating")
display(df.head())

Đọc file thành công!
Các cột trong file gốc: ['product_id', 'comment_id', 'title', 'thank_count', 'customer_id', 'rating', 'content']
Đường dẫn file: c:\Users\quang\Documents\BÀI TẬP\file_database\comments.csv\comments.csv
Cột bình luận: 'content' -> text | Cột số sao: 'rating' -> rating


,product_id,comment_id,title,thank_count,customer_id,rating,text
0,74021317,12559756,Cực kì hài lòng,313,22051463,5,"Có những người bước đến, họ lấp đầy hạnh phúc ..."
1,74021317,16979365,Cực kì hài lòng,6,27791831,5,"Thấy nhiều bạn chê tiki gói hàng quá, may sao ..."
2,74021317,14069617,Cực kì hài lòng,25,17748750,5,"Bìa cực xinh, tiki giao hàng nhanh, sách không..."
3,74021317,8569824,Cực kì hài lòng,57,410797,5,"Sách lúc nhận hình thức rất ổn, không cong vên..."
4,74021317,18368714,Cực kì hài lòng,0,28545286,5,Một cuốn sách rất đáng đọc về tình yêu thương ...


### 2.1. Kiểm tra kích thước dữ liệu

In [4]:
print(f"Kích thước dữ liệu ban đầu (dòng, cột): {df.shape}")
print(f"Tổng số mẫu: {df.shape[0]}")
print(f"Tổng số thuộc tính: {df.shape[1]}")
print("Các cột:", list(df.columns))
print()
df.info()

Kích thước dữ liệu ban đầu (dòng, cột): (141281, 7)
Tổng số mẫu: 141281
Tổng số thuộc tính: 7
Các cột: ['product_id', 'comment_id', 'title', 'thank_count', 'customer_id', 'rating', 'text']

<class 'pandas.DataFrame'>
RangeIndex: 141281 entries, 0 to 141280
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   product_id   141281 non-null  int64
 1   comment_id   141281 non-null  int64
 2   title        141277 non-null  str  
 3   thank_count  141281 non-null  int64
 4   customer_id  141281 non-null  int64
 5   rating       141281 non-null  int64
 6   text         103263 non-null  str  
dtypes: int64(5), str(2)
memory usage: 30.8 MB


### 2.2. Xác định và xử lý giá trị bị thiếu (NaN)

Ngoài ô trống (`NaN`), các bình luận chỉ gồm khoảng trắng cũng được coi là thiếu. Cột `rating` được đưa về dạng số (xử lý được cả dạng chữ như "5 sao"); những dòng có số sao không nằm trong khoảng 1–5 bị coi là lỗi và loại bỏ.

In [5]:
df["text"] = df["text"].replace(r"^\s*$", np.nan, regex=True)

df["rating"] = pd.to_numeric(
    df["rating"].astype(str).str.extract(r"(\d+(?:[.,]\d+)?)")[0].str.replace(",", ".", regex=False),
    errors="coerce",
)

missing_values = df[["text", "rating"]].isnull().sum()
print("Số giá trị thiếu theo từng cột (đã tính cả chuỗi rỗng):")
print(missing_values)

before = len(df)
df = df.dropna(subset=["text", "rating"])
log["Loại do thiếu text/rating (NaN)"] = before - len(df)

before = len(df)
df = df[df["rating"].between(1, 5)].copy()
df["rating"] = df["rating"].round().astype(int)
df["text"] = df["text"].astype(str)
log["Loại do số sao ngoài khoảng 1-5"] = before - len(df)

print(f"\nSố dòng còn lại sau khi xử lý thiếu và số sao không hợp lệ: {len(df)}")

Số giá trị thiếu theo từng cột (đã tính cả chuỗi rỗng):
text      38018
rating        0
dtype: int64

Số dòng còn lại sau khi xử lý thiếu và số sao không hợp lệ: 103263


### 2.3. Kiểm tra và xóa dữ liệu trùng lặp

Hai bình luận được coi là trùng nếu **cùng nội dung** (bỏ qua khác biệt chữ hoa/thường và khoảng trắng đầu/cuối) **và cùng số sao**. Các câu giống nhau nhưng khác số sao (mâu thuẫn nhãn) được thống kê riêng, không bị xóa.

In [6]:
tmp = df.assign(_key=df["text"].str.strip().str.lower())

duplicates_count = tmp.duplicated(subset=["_key", "rating"]).sum()
print(f"Số lượng dòng bị trùng lặp: {duplicates_count}")

conflict = tmp.groupby("_key")["rating"].nunique()
print(f"Số câu xuất hiện với nhiều mức sao khác nhau: {(conflict > 1).sum()}")

before = len(df)
df = df.loc[~tmp.duplicated(subset=["_key", "rating"])].reset_index(drop=True)
log["Loại do trùng lặp"] = before - len(df)
print(f"Kích thước sau khi loại bỏ dòng trùng lặp: {df.shape}")

Số lượng dòng bị trùng lặp: 27929
Số câu xuất hiện với nhiều mức sao khác nhau: 268
Kích thước sau khi loại bỏ dòng trùng lặp: (75334, 7)


### 2.4. Xác định các thuộc tính không cần thiết

In [7]:
keep_columns = ["text", "rating"]
unnecessary_columns = [c for c in df.columns if c not in keep_columns]
print(f"Các cột bị loại bỏ: {unnecessary_columns}")

df = df[keep_columns].copy()
print(f"Các cột còn lại: {list(df.columns)}")
df.head()

Các cột bị loại bỏ: ['product_id', 'comment_id', 'title', 'thank_count', 'customer_id']
Các cột còn lại: ['text', 'rating']


,text,rating
0,"Có những người bước đến, họ lấp đầy hạnh phúc ...",5
1,"Thấy nhiều bạn chê tiki gói hàng quá, may sao ...",5
2,"Bìa cực xinh, tiki giao hàng nhanh, sách không...",5
3,"Sách lúc nhận hình thức rất ổn, không cong vên...",5
4,Một cuốn sách rất đáng đọc về tình yêu thương ...,5


**Lý do loại bỏ:** các cột ngoài bình luận và số sao (mã sản phẩm, tên người dùng, thời gian, đường dẫn ảnh…) không dùng làm đầu vào hay nhãn cho bài toán phân loại cảm xúc, giữ lại chỉ gây dư thừa và nhiễu. **[ĐIỀN: liệt kê cụ thể các cột bị loại của nhóm]**

**Các cột giữ lại:** `text` (đầu vào) và `rating` (dùng để gán nhãn và vẽ biểu đồ phân bố sao).

### 2.5. Gán nhãn cảm xúc từ số sao

Quy tắc: 1–2 sao → **Negative (0)**, 3 sao → **Neutral (1)**, 4–5 sao → **Positive (2)**.

In [8]:
LABEL_NAMES = {0: "Negative", 1: "Neutral", 2: "Positive"}

def rating_to_label(r):
    if r <= 2:
        return 0
    if r == 3:
        return 1
    return 2

df["label"] = df["rating"].apply(rating_to_label)
df["sentiment"] = df["label"].map(LABEL_NAMES)
df[["text", "rating", "label", "sentiment"]].head()

,text,rating,label,sentiment
0,"Có những người bước đến, họ lấp đầy hạnh phúc ...",5,2,Positive
1,"Thấy nhiều bạn chê tiki gói hàng quá, may sao ...",5,2,Positive
2,"Bìa cực xinh, tiki giao hàng nhanh, sách không...",5,2,Positive
3,"Sách lúc nhận hình thức rất ổn, không cong vên...",5,2,Positive
4,Một cuốn sách rất đáng đọc về tình yêu thương ...,5,2,Positive


### 2.6. Chuẩn hóa văn bản (lowercase, loại bỏ ký tự đặc biệt/emoji)

Các bước trong hàm `normalize_text`:

1. Đồng nhất Unicode về dạng **NFC** (tránh lỗi chữ có dấu bị tách rời làm vỡ từ khi lọc ký tự).
2. Chuyển về **chữ thường** (lowercase).
3. Xóa **đường link (URL)**.
4. Rút gọn chữ bị **kéo dài** ("ngonnnn" → "ngon", "tốttttt" → "tốt"). Chỉ áp dụng cho chữ cái, số như "1000" được giữ nguyên.
5. Loại bỏ **emoji, dấu câu và ký tự đặc biệt**, chỉ giữ chữ cái (kể cả tiếng Việt), số và khoảng trắng.
6. Đổi một số **từ viết tắt phổ biến** trong review sang dạng đầy đủ ("ko" → "không", "dc" → "được", "sp" → "sản phẩm"…).
7. Xóa khoảng trắng thừa.

In [9]:
ABBREVIATIONS = {
    "ko": "không", "k": "không", "kh": "không", "hok": "không", "hông": "không",
    "dc": "được", "đc": "được",
    "sp": "sản phẩm", "mn": "mọi người", "vs": "với",
}

def normalize_text(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFC", text)
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"([^\W\d_])\1{2,}", r"\1", text)
    text = re.sub(r"[^\w\s]|_", " ", text)
    words = [ABBREVIATIONS.get(w, w) for w in text.split()]
    return " ".join(words)

### 2.7. Tách từ tiếng Việt (`underthesea`)

Tiếng Việt có nhiều từ ghép (ví dụ "sản phẩm", "chất lượng"), nên cần tách từ trước khi vector hóa. `word_tokenize(..., format="text")` nối các từ ghép bằng dấu gạch dưới: `chất lượng` → `chất_lượng`. Kết quả này cũng đúng định dạng đầu vào mà PhoBERT yêu cầu.

In [10]:
from underthesea import word_tokenize

def segment(text):
    return word_tokenize(normalize_text(text), format="text")

### 2.8. Loại bỏ từ dừng (stopwords)

Dùng danh sách stopwords tiếng Việt từ https://github.com/stopwords/vietnamese-stopwords (tự tải về nếu chưa có file).

**Lưu ý quan trọng cho bài toán cảm xúc:** danh sách này chứa cả những từ mang sắc thái cảm xúc hoặc phủ định như `tốt`, `hay`, `không`, `chưa`, `rất`, `quá`, `nhanh`, `lâu`... Nếu bỏ hết thì câu như "giao hàng không nhanh" hay "sản phẩm chưa tốt" sẽ mất đúng phần quyết định nhãn. Vì vậy các từ này được **giữ lại** (`KEEP_WORDS`).

Hai cột kết quả:
- `text_seg`: đã chuẩn hóa + tách từ, **chưa bỏ stopwords** (dùng cho PhoBERT).
- `text_clean`: đã bỏ stopwords (dùng cho TF-IDF + các mô hình truyền thống).

In [11]:
STOPWORDS_FILE = os.path.join(PROJECT_ROOT, "data", "raw", "vietnamese-stopwords.txt")
STOPWORDS_URL = "https://raw.githubusercontent.com/stopwords/vietnamese-stopwords/master/vietnamese-stopwords.txt"

FALLBACK_STOPWORDS = """và là của có cho các những một này đó thì mà ở với được bị bởi từ như khi nếu vì
nên để cũng đã sẽ đang vẫn còn lại ra vào lên xuống trong ngoài trên dưới sau trước rồi nhé nha ạ ơi
tôi mình bạn em anh chị ông bà cô cậu họ nó ta chúng_tôi chúng_ta thế sao gì đâu nào đây kia ấy
hay_là hoặc nhưng tuy_nhiên dù mặc_dù do bởi_vì cùng cả mọi từng tại theo về qua cứ chỉ""".split()

if not os.path.exists(STOPWORDS_FILE):
    try:
        urllib.request.urlretrieve(STOPWORDS_URL, STOPWORDS_FILE)
    except Exception as e:
        print("Không tải được danh sách stopwords (", e, ") -> dùng danh sách dự phòng. "
              "Nên bật Internet hoặc tự tải file về để dùng danh sách đầy đủ.")
        with open(STOPWORDS_FILE, "w", encoding="utf-8") as f:
            f.write("\n".join(FALLBACK_STOPWORDS))

KEEP_WORDS = {
    "không", "chưa", "chẳng", "chả", "đừng", "không_bao_giờ", "chưa_bao_giờ",
    "rất", "quá", "hơi", "khá", "thực_sự", "nhiều", "ít", "hơn", "thêm", "giảm", "tăng",
    "nên", "cần", "phải", "thiếu", "muốn",
    "tốt", "hay", "khó", "dễ", "đủ", "đúng", "rõ", "thích", "được", "nhanh", "lâu", "mới",
    "lớn", "nhỏ", "đẹp", "xấu", "kém", "tệ", "chậm", "rẻ", "đắt", "bền", "hỏng", "lỗi", "sai",
    "ưng", "hài_lòng", "thất_vọng", "tiếc", "ngon",
}

def load_stopwords(file_path, keep_words=KEEP_WORDS):
    with open(file_path, "r", encoding="utf-8") as f:
        sw = {unicodedata.normalize("NFC", line.strip().lower()).replace(" ", "_")
              for line in f if line.strip()}
    removed = sorted(sw & keep_words)
    print(f"Số stopwords trong file: {len(sw)}")
    print(f"Đã loại khỏi danh sách ({len(removed)} từ mang cảm xúc): {removed}")
    return sw - keep_words

def remove_stopwords(segmented_text, stopwords_set):
    return " ".join(w for w in segmented_text.split() if w not in stopwords_set)

stopwords_set = load_stopwords(STOPWORDS_FILE)
print("Số stopwords sau khi lọc:", len(stopwords_set))

Số stopwords trong file: 1942
Đã loại khỏi danh sách (33 từ mang cảm xúc): ['chưa', 'chưa_bao_giờ', 'cần', 'dễ', 'giảm', 'hay', 'hơn', 'khá', 'khó', 'không', 'không_bao_giờ', 'lâu', 'lớn', 'muốn', 'mới', 'nhanh', 'nhiều', 'nhỏ', 'nên', 'phải', 'quá', 'rõ', 'rất', 'thiếu', 'thêm', 'thích', 'thực_sự', 'tăng', 'tốt', 'ít', 'đúng', 'được', 'đủ']
Số stopwords sau khi lọc: 1909


### 2.9. Áp dụng lên toàn bộ dữ liệu

Những bình luận sau khi chuẩn hóa không còn nội dung chữ nào (ví dụ chỉ gồm emoji "👍👍👍") sẽ bị loại. Những bình luận bị bỏ hết chữ sau bước loại stopwords được dùng lại `text_seg` để không bị rỗng.

In [12]:
t0 = time.time()
try:
    from tqdm.auto import tqdm
    tqdm.pandas(desc="Tách từ")
    df["text_seg"] = df["text"].progress_apply(segment)
except ImportError:
    df["text_seg"] = df["text"].apply(segment)

before = len(df)
df = df[df["text_seg"].str.strip() != ""].reset_index(drop=True)
log["Loại do rỗng sau chuẩn hóa (chỉ emoji/ký tự đặc biệt)"] = before - len(df)

df["text_clean"] = df["text_seg"].apply(lambda t: remove_stopwords(t, stopwords_set))
print(f"Thời gian xử lý: {time.time() - t0:.1f} giây")

mask = df["text_clean"].str.strip() == ""
print("Số câu rỗng sau khi bỏ stopwords (đã dùng lại text_seg):", mask.sum())
df.loc[mask, "text_clean"] = df.loc[mask, "text_seg"]

before = len(df)
df = df.drop_duplicates(subset=["text_seg", "label"]).reset_index(drop=True)
log["Loại do trùng sau tiền xử lý"] = before - len(df)

print("Số từ trung bình - text gốc  :", round(df["text"].str.split().str.len().mean(), 2))
print("Số từ trung bình - text_seg  :", round(df["text_seg"].str.split().str.len().mean(), 2))
print("Số từ trung bình - text_clean:", round(df["text_clean"].str.split().str.len().mean(), 2))

pd.set_option("display.max_colwidth", 70, "display.width", 220)
df[["text", "text_seg", "text_clean", "rating", "label"]].head(8)

c:\Users\quang\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Tách từ: 100%|██████████| 75334/75334 [01:55<00:00, 654.85it/s] 


Thời gian xử lý: 115.6 giây
Số câu rỗng sau khi bỏ stopwords (đã dùng lại text_seg): 30
Số từ trung bình - text gốc  : 35.3
Số từ trung bình - text_seg  : 29.11
Số từ trung bình - text_clean: 16.85


,text,text_seg,text_clean,rating,label
0,"Có những người bước đến, họ lấp đầy hạnh phúc vào từng vết nứt tro...",có những người bước đến họ lấp đầy hạnh_phúc vào từng vết nứt tron...,lấp hạnh_phúc vết nứt tim được an_ủi chữa_lành đi tan_vỡ trái_tim ...,5,2
1,"Thấy nhiều bạn chê tiki gói hàng quá, may sao hàng mình không bị h...",thấy nhiều bạn chê tiki_gói hàng quá may_sao hàng mình không bị hư...,nhiều chê tiki_gói hàng quá may_sao hàng không hư tẹo nèo đúng hơi...,5,2
2,"Bìa cực xinh, tiki giao hàng nhanh, sách không bị nhăn góc, gãy cạ...",bìa cực xinh_tiki giao hàng nhanh sách không bị nhăn góc gãy cạnh ...,bìa cực xinh_tiki giao hàng nhanh sách không nhăn góc gãy cạnh hay...,5,2
3,"Sách lúc nhận hình thức rất ổn, không cong vênh xước gãy, đơn hàng...",sách lúc nhận hình_thức rất ổn không cong vênh_xước gãy đơn hàng n...,sách hình_thức rất ổn không cong vênh_xước gãy đơn hàng hài_lòng 5...,5,2
4,Một cuốn sách rất đáng đọc về tình yêu thương và sự mất mát mà chắ...,một cuốn sách rất đáng đọc về tình_yêu_thương và sự mất_mát mà chắ...,sách rất đọc tình_yêu_thương mất_mát gợi_ý rất hot cười trò nghịch...,5,2
5,"hic thấy người ta mua sách nhìn ổn áp ghê, tới mình mua đợt nào sá...",hic thấy người ta mua sách nhìn ổn_áp ghê tới mình mua đợt nào sác...,hic ta mua sách ổn_áp ghê mua đợt sách quăng góc trầy bìa lum_la k...,3,1
6,"Nội dung của sách thì không phải bàn đến rồi, đây là 1 tác phẩm vă...",nội_dung của sách thì không phải bàn đến rồi_đây là 1 tác_phẩm văn...,nội_dung sách không phải bàn 1 tác_phẩm văn_học kinh_điển nên dịch...,5,2
7,"Cho đến khi những trang sách cuối cùng ấy khép lại, tâm hồn tôi, t...",cho đến khi những trang_sách cuối_cùng ấy khép lại tâm_hồn tôi trá...,trang_sách khép tâm_hồn trái_tim bao thổn_thức một_chút đau_lòng m...,5,2


## 3. Kiểm thử dữ liệu (Data Testing)

Mục này kiểm tra chất lượng dữ liệu **sau khi làm sạch**, gồm hai nhiệm vụ: (1) dữ liệu không còn dòng lỗi/rỗng; (2) kiểm tra tỷ lệ phân phối nhãn Positive/Negative/Neutral. File kết quả chỉ được lưu khi **tất cả** các kiểm tra đều đạt.

### 3.1. Kiểm tra dữ liệu sau làm sạch có còn dòng lỗi/rỗng không

In [13]:
results = []

def check(name, condition):
    ok = bool(condition)
    results.append(ok)
    print(("PASS" if ok else "FAIL"), "-", name)

key = df["text"].str.strip().str.lower()

check("Không còn giá trị thiếu (NaN) ở text, rating, label",
      df[["text", "rating", "label", "text_seg", "text_clean"]].isnull().sum().sum() == 0)
check("Không còn bình luận gốc rỗng/chỉ có khoảng trắng",
      (df["text"].str.strip() == "").sum() == 0)
check("Không còn dòng rỗng ở text_seg",
      (df["text_seg"].str.strip() == "").sum() == 0)
check("Không còn dòng rỗng ở text_clean",
      (df["text_clean"].str.strip() == "").sum() == 0)
check("Số sao chỉ nằm trong khoảng 1-5",
      df["rating"].between(1, 5).all())
check("Nhãn chỉ gồm 0, 1, 2",
      set(df["label"].unique()) <= {0, 1, 2})
check("Nhãn khớp đúng quy tắc gán từ số sao",
      (df["rating"].apply(rating_to_label) == df["label"]).all())
check("Không còn dòng trùng lặp (cùng nội dung + cùng số sao)",
      not df.assign(_key=key).duplicated(subset=["_key", "rating"]).any())
check("text_clean không còn emoji/dấu câu/ký tự đặc biệt",
      not df["text_clean"].str.contains(r"[^\w\s]", regex=True).any())
check("text_clean đã được chuyển về chữ thường",
      (df["text_clean"] == df["text_clean"].str.lower()).all())

print(f"\nKết quả: {sum(results)}/{len(results)} kiểm tra đạt")
assert all(results), "Có kiểm tra không đạt - hãy xem các dòng FAIL ở trên trước khi lưu file."

PASS - Không còn giá trị thiếu (NaN) ở text, rating, label
PASS - Không còn bình luận gốc rỗng/chỉ có khoảng trắng
PASS - Không còn dòng rỗng ở text_seg
PASS - Không còn dòng rỗng ở text_clean
PASS - Số sao chỉ nằm trong khoảng 1-5
PASS - Nhãn chỉ gồm 0, 1, 2
PASS - Nhãn khớp đúng quy tắc gán từ số sao
PASS - Không còn dòng trùng lặp (cùng nội dung + cùng số sao)
FAIL - text_clean không còn emoji/dấu câu/ký tự đặc biệt
PASS - text_clean đã được chuyển về chữ thường

Kết quả: 9/10 kiểm tra đạt


AssertionError: Có kiểm tra không đạt - hãy xem các dòng FAIL ở trên trước khi lưu file.

### 3.2. Kiểm tra tỷ lệ phân phối nhãn (Positive / Negative / Neutral)

In [ ]:
counts = df["sentiment"].value_counts().reindex(["Negative", "Neutral", "Positive"]).fillna(0).astype(int)
pct = (counts / counts.sum() * 100).round(2)
label_table = pd.DataFrame({"Số mẫu": counts, "Tỷ lệ (%)": pct})
print(label_table)

print("\nPhân bố theo số sao (để đối chiếu):")
star_counts = df["rating"].value_counts().sort_index()
print(pd.DataFrame({"Số mẫu": star_counts, "Tỷ lệ (%)": (star_counts / star_counts.sum() * 100).round(2)}))

assert (counts > 0).all(), "Có nhãn không có mẫu nào - không thể huấn luyện phân loại 3 lớp."
if pct.min() < 10:
    print(f"\nCẢNH BÁO: lớp '{pct.idxmin()}' chỉ chiếm {pct.min()}% -> dữ liệu mất cân bằng, "
          "cần dùng F1-macro và/hoặc xử lý mất cân bằng (class_weight, lấy mẫu lại) khi huấn luyện.")
ratio = counts.max() / counts.min()
print(f"Tỷ lệ lớp lớn nhất / lớp nhỏ nhất: {ratio:.1f} lần")

           Số mẫu  Tỷ lệ (%)
sentiment                   
Negative     5444       7.40
Neutral      4240       5.76
Positive    63903      86.84

Phân bố theo số sao (để đối chiếu):
        Số mẫu  Tỷ lệ (%)
rating                   
1         3353       4.56
2         2091       2.84
3         4240       5.76
4         8715      11.84
5        55188      75.00

CẢNH BÁO: lớp 'Neutral' chỉ chiếm 5.76% -> dữ liệu mất cân bằng, cần dùng F1-macro và/hoặc xử lý mất cân bằng (class_weight, lấy mẫu lại) khi huấn luyện.
Tỷ lệ lớp lớn nhất / lớp nhỏ nhất: 15.1 lần


In [ ]:
# Sinh sẵn đoạn nhận xét với số liệu thật để dán vào mục "Nhận xét" bên dưới
neg, neu, pos = (int(counts[k]) for k in ["Negative", "Neutral", "Positive"])
tot = neg + neu + pos
state = "mất cân bằng" if pct.min() < 10 or ratio > 3 else "tương đối cân bằng"
print(f"Nhận xét: lớp Negative có {neg:,} mẫu ({neg/tot*100:.2f}%), Neutral có {neu:,} mẫu ({neu/tot*100:.2f}%) "
      f"và Positive có {pos:,} mẫu ({pos/tot*100:.2f}%). Lớp lớn nhất gấp {ratio:.1f} lần lớp nhỏ nhất, "
      f"nên dữ liệu {state}.")
print(f"Tổng số mẫu sau làm sạch: {len(df):,} (ban đầu {log['Số dòng ban đầu']:,}).")

Nhận xét: lớp Negative có 5,444 mẫu (7.40%), Neutral có 4,240 mẫu (5.76%) và Positive có 63,903 mẫu (86.84%). Lớp lớn nhất gấp 15.1 lần lớp nhỏ nhất, nên dữ liệu mất cân bằng.
Tổng số mẫu sau làm sạch: 73,587 (ban đầu 141,281).


**Nhận xét:** lớp Negative có **[ĐIỀN số mẫu]**, Neutral có **[ĐIỀN số mẫu]** và Positive có **[ĐIỀN số mẫu]**. Lớp Neutral chiếm khoảng **[ĐIỀN]%**, nên dữ liệu **[ĐIỀN: mất cân bằng / tương đối cân bằng]**.

**Ý nghĩa với bài toán:** nếu dữ liệu mất cân bằng thì Accuracy không phản ánh đúng chất lượng mô hình (mô hình chỉ cần đoán tốt lớp lớn đã có Accuracy cao), vì vậy các thành viên huấn luyện mô hình cần dùng thêm F1-macro, theo dõi riêng từng lớp và cân nhắc xử lý mất cân bằng.

### 3.3. Bảng tổng hợp quá trình làm sạch

In [ ]:
summary = pd.DataFrame({"Số dòng": log})
summary.loc["Số dòng cuối cùng (sau làm sạch)"] = len(df)
summary.loc["Tổng số dòng bị loại"] = log["Số dòng ban đầu"] - len(df)
summary["Tỷ lệ so với ban đầu (%)"] = (summary["Số dòng"] / log["Số dòng ban đầu"] * 100).round(2)
summary

,Số dòng,Tỷ lệ so với ban đầu (%)
Số dòng ban đầu,141281,100.00
Loại do thiếu text/rating (NaN),38018,26.91
Loại do số sao ngoài khoảng 1-5,0,0.00
Loại do trùng lặp,27929,19.77
Loại do rỗng sau chuẩn hóa (chỉ emoji/ký tự đặc biệt),92,0.07
Loại do trùng sau tiền xử lý,1655,1.17
Số dòng cuối cùng (sau làm sạch),73587,52.09
Tổng số dòng bị loại,67694,47.91


### 3.4. Xem thử một số mẫu trước và sau khi xử lý

In [ ]:
df[["text", "text_clean", "rating", "sentiment"]].sample(min(10, len(df)), random_state=42)

,text,text_clean,rating,sentiment
18441,"Một trong những quyển sách hay nhất về cách cải thiện thói quen, g...",quyển sách hay cải_thiện thói_quen giúp phát_triển 10 10,5,Positive
27613,"Đóng gói kỹ giao hàng sớm hơn dự kiến.\r\nSách in đẹp, rõ.\r\nCảm ...",đóng_gói kỹ giao hàng hơn dự_kiến sách in đẹp rõ tác_giả shop,5,Positive
48594,"Sách cầm chắc tay, hình in sắc sảo, rõ nét, sinh động",sách cầm_chắc hình_in sắc_sảo rõ nét sinh_động,5,Positive
60540,Đóng gói sơ sài quá sách mà k có chống sốc gì hết :(((,đóng_gói sơ_sài quá sách không chống sốc,5,Positive
45694,Đóng gói sách bên tiki không chèn mút sốp nên phần góc bị cong. Gi...,đóng gói sách tiki không chèn mút sốp nên góc cong giao hàng nhanh...,4,Positive
70098,"mua được với giá giảm 50% + tiki xu + zalopay còn 85k, sách quá đẹ...",mua được giá giảm 50 tiki xu_zalopay 85 k sách quá đẹp mê đẹp quá ...,5,Positive
50593,"Nội dung sách khá hay, có khả năng khiến cho độc giả có thêm những...",nội_dung sách khá hay khả_năng độc_giả thêm thông_tin suy_nghĩ bộ_...,5,Positive
22780,Mình là người ít đọc sách và đọc chậm. Nhưng riêng cuốn này đọc tr...,ít đọc sách đọc chậm đọc phương_pháp tác_giả rất hữu_ích đọc nhanh...,5,Positive
11455,Tôi cho 3* thôi vì tiki hủy đơn có gift poster lụa 86 nên tôi khôn...,3 tiki hủy đơn gift poster lụa 86 nên không hài_lòng chưa sách khô...,3,Neutral
18985,Nhiều cuốn sách về giáo dục con thường chỉ tập trung vào phương ph...,nhiều sách giáo_dục phương_pháp quên gốc_rễ quên đứa trẻ cá_thể ri...,5,Positive


### 3.5. Lưu file kết quả (bàn giao cho Thành viên 2 và các thành viên huấn luyện mô hình)

In [ ]:
OUT_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "reviews_clean.csv")
out_cols = ["text", "text_seg", "text_clean", "rating", "label", "sentiment"]
df[out_cols].to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print("Đã lưu:", os.path.abspath(OUT_PATH), "| kích thước:", df[out_cols].shape)
print("Các cột:", out_cols)

Đã lưu: c:\Users\Admin\Downloads\file_database\reviews_clean.csv | kích thước: (73587, 6)
Các cột: ['text', 'text_seg', 'text_clean', 'rating', 'label', 'sentiment']


**Ý nghĩa các cột trong file bàn giao `reviews_clean.csv`:**

| Cột | Dùng cho |
|---|---|
| `text` | Văn bản gốc – hiển thị, thống kê độ dài câu (biểu đồ độ dài bình luận theo cảm xúc). |
| `text_seg` | Văn bản đã chuẩn hóa + tách từ, giữ stopwords – dùng cho PhoBERT. |
| `text_clean` | Văn bản đã bỏ stopwords – dùng cho TF-IDF, WordCloud, biểu đồ tần suất từ khóa. |
| `rating` | Số sao 1–5 – biểu đồ phân bố điểm đánh giá sao. |
| `label` / `sentiment` | Nhãn cảm xúc dạng số (0/1/2) và dạng chữ – nhãn huấn luyện mô hình. |

In [20]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
import pickle

# 1. Đọc dữ liệu
df = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'processed', 'reviews_clean.csv'))

# 2. Lấy dữ liệu text và nhãn (Lưu ý: Nếu cột nhãn của nhóm bạn không phải 'sentiment', hãy thay thế bằng tên cột thực tế)
X = df['text'].astype(str).values
y = np.array(df['sentiment'])  # Dùng np.array() để loại bỏ lỗi pyarrow

# 3. Vector hóa văn bản
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_vect = vectorizer.fit_transform(X)

# 4. Chia tập Train / Test an toàn
X_train, X_test, y_train, y_test = train_test_split(
    X_vect, y, test_size=0.2, random_state=42
)

print('✅ Chia tập train/test thành công!')
print(f'- Số lượng mẫu train: {X_train.shape[0]}')
print(f'- Số lượng mẫu test: {X_test.shape[0]}')

# 5. Huấn luyện và lưu lại mô hình
model_lr = LogisticRegression()
model_lr.fit(X_train, y_train)

model_nb = MultinomialNB()
model_nb.fit(X_train, y_train)

# Lưu file .pkl
with open(os.path.join(PROJECT_ROOT, 'models', 'tfidf_vectorizer.pkl'), 'wb') as f:
  pickle.dump(vectorizer, f)

with open(os.path.join(PROJECT_ROOT, 'models', 'logistic_regression_model.pkl'), 'wb') as f:
  pickle.dump(model_lr, f)

with open(os.path.join(PROJECT_ROOT, 'models', 'naive_bayes_model.pkl'), 'wb') as f:
  pickle.dump(model_nb, f)

print('✅ Huấn luyện và lưu tất cả các file .pkl thành công!')

✅ Chia tập train/test thành công!
- Số lượng mẫu train: 58869
- Số lượng mẫu test: 14718
✅ Huấn luyện và lưu tất cả các file .pkl thành công!
